In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from IPython.display import display
import math

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


## Group Metadata Notebook:
#### We want to create recording groups to be able to do stats on our spike parameterization to figure out what metadata parameters result in spike waveform differences. We need to create groups for this comparison because because each recording has multiple variables (species, age, sex, brainorigin, somalayer, dendritic type), that varies for each recording. Thus, when we ran the stats for spike params prior to the group separation, we didn't know which metadata params were generating the differences/influencing the changes in spike waveform.

#### I initially was doing the grouping manually, but now doing automized version (below)

#### Note: I am ignoring age (all are adults?)and weight metadata params (for now)

#### Helper functions 

In [2]:
# Functions to save a datrame to a pickle file and another to extract the data from the pickle file

def save_dataframe_to_pickle(dataframe, file_path):
    """
    Function to save a DataFrame as a pickle file.
    
    Args:
    - dataframe (pd.DataFrame): DataFrame to be saved.
    - file_path (str): Path to save the pickle file.
    """
    dataframe.to_pickle(file_path)
    print(f"Data frame saved to {file_path}")

def load_dataframe_from_pickle(file_path):
    """
    Function to extract a DataFrame from a pickle file.
    
    Args:
    - file_path (str): Path to the pickle file.
    
    Returns:
    - dataframe (pd.DataFrame): Loaded DataFrame.
    """
    dataframe = pd.read_pickle(file_path)
    return dataframe

def save_dict_to_pickle(dictionary, filepath):
    """
    Save a dictionary to a pickle file.

    Parameters:
        dictionary (dict): The dictionary to save.
        filepath (str): The path to the pickle file.
    """
    with open(filepath, 'wb') as f:
        pickle.dump(dictionary, f)
    print(f"Dictionary saved to {filepath}")


def load_dict_from_pickle(filepath):
    """
    Load a dictionary from a pickle file.

    Parameters:
        filepath (str): The path to the pickle file.

    Returns:
        dict: The loaded dictionary.
    """
    with open(filepath, 'rb') as f:
        dictionary = pickle.load(f)
    print(f"Dictionary loaded from {filepath}")
    return dictionary

### Read in pickle with filtered parameterized data
#### This pandas dataframe contains spike param data and animal/cell metadata for each spike(one spike per row). The dataframe has already been filtered for rsq fits (see other notebooks for rsq thresholds)

In [3]:
allMonkey_df = load_dataframe_from_pickle(r"/Users/blancamartin/Downloads/allMonkey_df_filt (1).pkl")

In [4]:
allMonkey_df

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,...,Spike_#,Spike_ID,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species
1,1.832248,0.60,-38.827516,11.984253,0.55,3.929138,5.981717,-53.646425,56.35,0.950116,...,1,M03f0SwSweep_10Sp1,M03_JS_A1_C01,A,3.0,PFC,6.3,F,9.96,Macaca fascicularis
2,1.725643,0.60,-38.571168,11.453247,0.55,3.533936,6.194347,-53.275150,56.20,0.969743,...,2,M03f0SwSweep_10Sp2,M03_JS_A1_C01,A,3.0,PFC,6.3,F,9.96,Macaca fascicularis
3,1.762723,0.60,-39.727784,11.669922,0.60,3.567505,5.819483,-53.277108,NaN,0.975530,...,3,M03f0SwSweep_10Sp3,M03_JS_A1_C01,A,3.0,PFC,6.3,F,9.96,Macaca fascicularis
5,2.058032,0.60,-39.135743,11.889649,0.55,3.807068,6.171128,-52.532920,64.70,0.963031,...,1,M03f0SwSweep_11Sp1,M03_JS_A1_C01,A,3.0,PFC,6.3,F,9.96,Macaca fascicularis
7,2.350725,0.60,-38.891602,11.648560,0.60,3.855896,6.057973,-52.542671,67.35,0.992663,...,3,M03f0SwSweep_11Sp3,M03_JS_A1_C01,A,3.0,PFC,6.3,F,9.96,Macaca fascicularis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12545,1.914118,1.35,-42.074463,44.442871,2.50,0.457764,0.338319,-81.006579,NaN,0.958351,...,0,M21f12SwSweep_8Sp0,M21_SA_A1_C07,S,3.0,V1,8.7,M,7.65,Macaca fascicularis
12546,1.846199,1.35,-42.471191,45.846680,2.45,0.442505,0.359868,-78.247398,121.70,0.923707,...,0,M21f12SwSweep_9Sp0,M21_SA_A1_C07,S,3.0,V1,8.7,M,7.65,Macaca fascicularis
12547,2.099518,1.45,-39.785645,42.367676,2.55,0.381470,0.299768,-84.266822,255.00,0.944645,...,1,M21f12SwSweep_9Sp1,M21_SA_A1_C07,S,3.0,V1,8.7,M,7.65,Macaca fascicularis
12548,2.276198,1.55,-37.100098,40.139893,2.65,0.335693,0.304954,-81.487302,359.55,0.930923,...,2,M21f12SwSweep_9Sp2,M21_SA_A1_C07,S,3.0,V1,8.7,M,7.65,Macaca fascicularis


In [5]:
test = allMonkey_df['dendriticType'].unique()

In [6]:
test

array(['A', 'unknown', 'S'], dtype=object)

In [7]:
display(allMonkey_df.loc[allMonkey_df['dendriticType'].isna()])

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,...,Spike_#,Spike_ID,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species


### Create groups

In [8]:


# Identify metadata columns (excluding 'Monkey ID')
metadata_cols = ['dendriticType', 'SomaLayerLoc', 'brainOrigin', 'Sex', 'Species']


# Function to create subcomparisons for varying metadata
def create_comparison_groups(df, metadata_cols):
    results = []

    for varying_col in metadata_cols:
        # Columns to keep fixed (all except the varying one)
        fixed_cols = [col for col in metadata_cols if col != varying_col]

        # Group data by fixed metadata columns
        grouped = df.groupby(fixed_cols)
        subcomparison_index = 1

        for group_name, group_data in grouped:
            # Get unique values for the varying column within the fixed group
            varying_groups = group_data[varying_col].unique()

            # Ensure there are at least two values for the varying column
            if len(varying_groups) > 1:
                # Create a single subcomparison that includes all varying groups
                subsets = {value: group_data[group_data[varying_col] == value] for value in varying_groups}
                
                results.append({
                    'Subcomparison': subcomparison_index,
                    'Varying Metadata': varying_col,
                    'Fixed Metadata': ', '.join([f"{col}={val}" for col, val in zip(fixed_cols, group_name)]) if isinstance(group_name, tuple) else f"{fixed_cols[0]}={group_name}",
                    'Groups': {value: len(subset) for value, subset in subsets.items()}
                })

                subcomparison_index += 1

    return pd.DataFrame(results)

# Run the function to create comparison groups
grouping_results_df = create_comparison_groups(allMonkey_df, metadata_cols)

# Display the table in chunks for better readability
from IPython.display import display

# Display the table in chunks if it is too large
if len(grouping_results_df) > 50:
    for i in range(0, len(grouping_results_df), 50):
        display(grouping_results_df.iloc[i:i+50])
else:
    display(grouping_results_df)



,Subcomparison,Varying Metadata,Fixed Metadata,Groups
0,1,dendriticType,"SomaLayerLoc=2.0, brainOrigin=PFC, Sex=F, Spec...","{'A': 74, 'S': 85}"
1,2,dendriticType,"SomaLayerLoc=2.0, brainOrigin=PFC, Sex=M, Spec...","{'S': 22, 'A': 139}"
2,3,dendriticType,"SomaLayerLoc=3.0, brainOrigin=PFC, Sex=F, Spec...","{'A': 115, 'S': 675, 'unknown': 219}"
3,4,dendriticType,"SomaLayerLoc=3.0, brainOrigin=PFC, Sex=M, Spec...","{'A': 343, 'S': 202}"
4,5,dendriticType,"SomaLayerLoc=3.0, brainOrigin=V1, Sex=M, Speci...","{'S': 598, 'unknown': 9, 'A': 522}"
5,6,dendriticType,"SomaLayerLoc=3.0, brainOrigin=V1, Sex=M, Speci...","{'S': 64, 'A': 150, 'unknown': 21}"
6,7,dendriticType,"SomaLayerLoc=4.0, brainOrigin=PFC, Sex=F, Spec...","{'S': 192, 'A': 139}"
7,8,dendriticType,"SomaLayerLoc=4.0, brainOrigin=PFC, Sex=M, Spec...","{'A': 74, 'S': 114}"
8,9,dendriticType,"SomaLayerLoc=4.0, brainOrigin=V1, Sex=M, Speci...","{'unknown': 3, 'S': 135, 'A': 8}"
9,10,dendriticType,"SomaLayerLoc=2_3, brainOrigin=V1, Sex=M, Speci...","{'S': 134, 'A': 80}"


In [9]:
grouping_results_df = grouping_results_df.drop([1, 8, 14, 20, 21, 24, 30, 35])

In [10]:
display(grouping_results_df)

,Subcomparison,Varying Metadata,Fixed Metadata,Groups
0,1,dendriticType,"SomaLayerLoc=2.0, brainOrigin=PFC, Sex=F, Spec...","{'A': 74, 'S': 85}"
2,3,dendriticType,"SomaLayerLoc=3.0, brainOrigin=PFC, Sex=F, Spec...","{'A': 115, 'S': 675, 'unknown': 219}"
3,4,dendriticType,"SomaLayerLoc=3.0, brainOrigin=PFC, Sex=M, Spec...","{'A': 343, 'S': 202}"
4,5,dendriticType,"SomaLayerLoc=3.0, brainOrigin=V1, Sex=M, Speci...","{'S': 598, 'unknown': 9, 'A': 522}"
5,6,dendriticType,"SomaLayerLoc=3.0, brainOrigin=V1, Sex=M, Speci...","{'S': 64, 'A': 150, 'unknown': 21}"
6,7,dendriticType,"SomaLayerLoc=4.0, brainOrigin=PFC, Sex=F, Spec...","{'S': 192, 'A': 139}"
7,8,dendriticType,"SomaLayerLoc=4.0, brainOrigin=PFC, Sex=M, Spec...","{'A': 74, 'S': 114}"
9,10,dendriticType,"SomaLayerLoc=2_3, brainOrigin=V1, Sex=M, Speci...","{'S': 134, 'A': 80}"
10,11,dendriticType,"SomaLayerLoc=5_6, brainOrigin=PFC, Sex=F, Spec...","{'S': 387, 'unknown': 140, 'A': 193}"
11,12,dendriticType,"SomaLayerLoc=5_6, brainOrigin=PFC, Sex=M, Spec...","{'A': 543, 'S': 117}"


In [11]:
grouping_results_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39 entries, 0 to 46
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Subcomparison     39 non-null     int64 
 1   Varying Metadata  39 non-null     object
 2   Fixed Metadata    39 non-null     object
 3   Groups            39 non-null     object
dtypes: int64(1), object(3)
memory usage: 1.5+ KB


In [13]:
save_dataframe_to_pickle(grouping_results_df, r"/Users/blancamartin/Downloads/groupingResults.pkl")

Data frame saved to /Users/blancamartin/Downloads/groupingResults.pkl


In [14]:
Metadata = ["brainOrigin", "Sex", "dendriticType", "SomaLayerLoc", "Species"]
for meta in Metadata:
    print(allMonkey_df[meta].unique())
    print(allMonkey_df[meta].nunique())

['PFC' 'IC' 'V1' 'LIP']
4
['F' 'M']
2
['A' 'unknown' 'S']
3
[3.0 'unknown' 2.0 4.0 '5_6' '2_3']
6
['Macaca fascicularis' 'Macaca mulatta']
2
